# Task 09 – Interactive AI Dashboard Prototype (Streamlit)
### SmartCare Hospital AI Dataset | Option C – Disease Risk Classification
### CCS3440 Artificial Intelligence Coursework

* **Author:** Siluna Nusal (CIT-23-02-0025) | `@GitGuru29`
* **Module:** CCS3440 – Artificial Intelligence
* **Target Variable:** `disease_risk_level` (0 = Low, 1 = Medium, 2 = High)

---

### Prototype Deployment Protocol
This notebook deploys the Champion Model (selected in Task 06) as an interactive clinical decision-support dashboard using **Streamlit**.
* The application runs directly within the Google Colab environment.
* It captures patient clinical and demographic parameters.
* It provides real-time risk stratification (Low, Medium, High).
* It integrates Local SHAP Explainability (from Task 07) to provide the physician with a transparent reasoning for the prediction.

In [ ]:
# 1. Install Required XAI and Dashboard Libraries
!pip install -q streamlit shap joblib

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Path Configuration & Environment Setup
import os
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/AI Assignment')
MODELS_DIR = BASE_DIR / 'models'
DATA_DIR = BASE_DIR / 'processed'

X_TRAIN_PATH = DATA_DIR / 'X_train.csv'

# Load the Champion Model dynamically based on Task 06 selection
try:
    with open(MODELS_DIR / 'final_model_selection.json') as f:
        selection = json.load(f)
    MODEL_PATH = MODELS_DIR / selection['best_model_file']
    model_name = selection['best_model']
    print(f" Champion Model path identified: {MODEL_PATH} ({model_name})")
except FileNotFoundError:
    print(" Warning: final_model_selection.json not found! Using Logistic Regression as fallback.")
    MODEL_PATH = MODELS_DIR / 'logistic_regression.pkl'
    model_name = "Logistic Regression"

# Verify files exist
if not os.path.exists(MODEL_PATH):
    print(f" Error: Model file not found at {MODEL_PATH}")
if not os.path.exists(X_TRAIN_PATH):
    print(f" Error: Training data not found at {X_TRAIN_PATH}")
else:
    print(f" Training data successfully located at {X_TRAIN_PATH}")

 Champion Model path identified: /content/drive/MyDrive/AI Assignment/models/logistic_regression.pkl (Logistic Regression)
 Training data successfully located at /content/drive/MyDrive/AI Assignment/processed/X_train.csv


In [ ]:
%%writefile /content/app.py

import os
import warnings
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# PAGE CONFIGURATION

st.set_page_config(
    page_title="SmartCare AI | Disease Risk Stratification",
    page_icon="",
    layout="wide"
)


# HARDCODED PATHS (Inside Colab)

MODEL_PATH = "/content/drive/MyDrive/AI Assignment/models/logistic_regression.pkl"
DATA_PATH = "/content/drive/MyDrive/AI Assignment/processed/X_train.csv"
METADATA_PATH = "/content/drive/MyDrive/AI Assignment/models/final_model_selection.json"


# CACHED LOADERS

@st.cache_resource
def load_champion_model():
    model = joblib.load(MODEL_PATH)
    if not hasattr(model, 'multi_class'):
        model.multi_class = 'ovr'
    return model

@st.cache_data
def load_feature_space():
    X_train = pd.read_csv(DATA_PATH)
    return X_train.astype('float64')

try:
    model = load_champion_model()
    X_background = load_feature_space()
except Exception as e:
    st.error(" Critical Error: Could not load the SmartCare clinical model or background data.")
    st.exception(e)
    st.stop()


# HEADER

st.title(" SmartCare AI Clinical Dashboard")
st.subheader("Disease Risk Stratification Engine (Option C)")
st.caption("Powered by Machine Learning & Shapley Additive Explanations (SHAP)")
st.divider()


# SIDEBAR

st.sidebar.title("System Diagnostics")
st.sidebar.write("Clinical Decision Support System (CDSS) for triaging patients into Low, Medium, or High disease risk tiers.")
st.sidebar.divider()
st.sidebar.write("**Deployed Algorithm:** Logistic Regression")
st.sidebar.write(f"**Total Features:** {X_background.shape[1]}")
st.sidebar.write("**Explainability Core:** SHAP LinearExplainer")


# CLINICAL INPUT FORM

st.header(" Patient Clinical Profile")

col1, col2, col3 = st.columns(3)

with col1:
    age = st.number_input("Age (Years)", min_value=0, max_value=120, value=45, step=1)
    gender = st.selectbox("Gender", ["Male", "Female"])
    blood_group = st.selectbox("Blood Group", ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"])

with col2:
    systolic_bp = st.number_input("Systolic Blood Pressure (mmHg)", min_value=50.0, max_value=250.0, value=120.0)
    diastolic_bp = st.number_input("Diastolic Blood Pressure (mmHg)", min_value=30.0, max_value=150.0, value=80.0)
    blood_sugar = st.number_input("Fasting Blood Sugar (mg/dL)", min_value=50.0, max_value=500.0, value=95.0)

with col3:
    cholesterol = st.number_input("Cholesterol (mg/dL)", min_value=50.0, max_value=500.0, value=180.0)
    bmi = st.number_input("Body Mass Index (BMI)", min_value=10.0, max_value=80.0, value=24.5)
    previous_admissions = st.number_input("Previous Hospital Admissions", min_value=0, max_value=50, value=0, step=1)

health_burden_score = st.slider("Calculated Health Burden Score", min_value=0.0, max_value=100.0, value=15.0)

st.divider()

# FEATURE ENGINEERING & VECTORIZATION

def construct_feature_vector():
    row = pd.DataFrame(0.0, index=[0], columns=X_background.columns)

    row.loc[0, 'age'] = age
    row.loc[0, 'systolic_bp'] = systolic_bp
    row.loc[0, 'diastolic_bp'] = diastolic_bp
    row.loc[0, 'blood_sugar_mg_dl'] = blood_sugar
    row.loc[0, 'cholesterol_mg_dl'] = cholesterol
    row.loc[0, 'bmi'] = bmi
    row.loc[0, 'previous_admissions'] = previous_admissions
    row.loc[0, 'health_burden_score'] = health_burden_score

    if f'gender_{gender}' in row.columns:
        row.loc[0, f'gender_{gender}'] = 1.0

    if f'blood_group_{blood_group}' in row.columns:
        row.loc[0, f'blood_group_{blood_group}'] = 1.0

    if bmi < 18.5: bmi_cat = 'Underweight'
    elif bmi < 25: bmi_cat = 'Normal'
    elif bmi < 30: bmi_cat = 'Overweight'
    else: bmi_cat = 'Obese'
    if f'bmi_category_{bmi_cat}' in row.columns: row.loc[0, f'bmi_category_{bmi_cat}'] = 1.0

    if blood_sugar < 100: bs_cat = 'Normal'
    elif blood_sugar < 126: bs_cat = 'Prediabetic'
    else: bs_cat = 'Diabetic'
    if f'blood_sugar_category_{bs_cat}' in row.columns: row.loc[0, f'blood_sugar_category_{bs_cat}'] = 1.0

    return row


# PREDICTION PROTOCOL

predict_button = st.button("Generate Diagnostic Risk Assessment", type="primary", use_container_width=True)

if predict_button:
    try:
        input_vector = construct_feature_vector()

        predicted_idx = model.predict(input_vector)[0]
        probabilities = model.predict_proba(input_vector)[0]

        risk_map = {0: 'LOW RISK', 1: 'MEDIUM RISK', 2: 'HIGH RISK'}
        final_risk = risk_map[predicted_idx]

        st.subheader("Diagnosis Result:")
        if predicted_idx == 0:
            st.success(f" Patient Status: {final_risk}")
        elif predicted_idx == 1:
            st.warning(f" Patient Status: {final_risk}")
        else:
            st.error(f" Patient Status: {final_risk}")

        st.write("### Confidence Probabilities")
        c1, c2, c3 = st.columns(3)
        c1.metric("Low Risk Probability", f"{probabilities[0]*100:.1f}%")
        c2.metric("Medium Risk Probability", f"{probabilities[1]*100:.1f}%")
        c3.metric("High Risk Probability", f"{probabilities[2]*100:.1f}%")

        st.divider()

        # ============================================================
        # SHAP LOCAL EXPLANATION (GLASS-BOX AI) - 1D Bulletproof Fix
        # ============================================================
        st.header(" Explainable AI (SHAP) - Clinical Justification")
        st.write(f"Analyzing the specific factors pushing the patient towards **{final_risk}** classification.")

        with st.spinner("Executing SHAP Force Analysis..."):
            explainer = shap.LinearExplainer(model, shap.sample(X_background, 100, random_state=42))
            shap_vals = explainer.shap_values(input_vector)

            # 1. Safely Extract the proper Array based on SHAP version
            if isinstance(shap_vals, list):
                # Older SHAP: List of Arrays
                raw_shap = shap_vals[predicted_idx][0]
            else:
                # Newer SHAP: Object or Array
                if hasattr(shap_vals, 'values'):
                    shap_vals = shap_vals.values
                shap_vals = np.array(shap_vals)

                if shap_vals.ndim == 3:
                    raw_shap = shap_vals[0, :, predicted_idx]
                elif shap_vals.ndim == 2:
                    raw_shap = shap_vals[0]
                else:
                    raw_shap = shap_vals

            # 2. Force conversion to Native 1D Python Lists (Prevents Pandas ValueError)
            class_shap_values = np.ravel(raw_shap).tolist()
            patient_vals = np.ravel(input_vector.iloc[0].values).tolist()
            feature_names = input_vector.columns.tolist()

            # 3. Build DataFrame
            impact_df = pd.DataFrame({
                "Clinical Feature": feature_names,
                "Patient's Value": patient_vals,
                "SHAP Impact": class_shap_values
            })

            impact_df['Absolute Impact'] = np.abs(impact_df['SHAP Impact'])
            top_drivers = impact_df.sort_values('Absolute Impact', ascending=False).head(10)

            top_drivers['Direction'] = top_drivers['SHAP Impact'].apply(lambda x: "Increases Risk" if x > 0 else "Decreases Risk")

            st.dataframe(top_drivers[["Clinical Feature", "Patient's Value", "SHAP Impact", "Direction"]], use_container_width=True, hide_index=True)

        st.warning(
            " **Ethical & Safety Disclaimer:**\n"
            "This CDSS prototype is developed strictly for academic demonstration. "
            "Algorithmic predictions are statistically derived and MUST NOT overrule "
            "certified medical judgment."
        )

    except Exception as e:
        st.error(" Diagnostic Pipeline Failed.")
        st.exception(e)

Overwriting /content/app.py


In [ ]:
%%writefile /content/app.py

import os
import warnings
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# PAGE CONFIGURATION
# ============================================================
st.set_page_config(
    page_title="SmartCare AI | Disease Risk Stratification",
    page_icon="",
    layout="wide"
)

# ============================================================
# HARDCODED PATHS (Inside Colab)
# ============================================================
MODEL_PATH = "/content/drive/MyDrive/AI Assignment/models/logistic_regression.pkl"
DATA_PATH = "/content/drive/MyDrive/AI Assignment/processed/X_train.csv"
METADATA_PATH = "/content/drive/MyDrive/AI Assignment/models/final_model_selection.json"

# ============================================================
# CACHED LOADERS
# ============================================================
@st.cache_resource
def load_champion_model():
    model = joblib.load(MODEL_PATH)
    if not hasattr(model, 'multi_class'):
        model.multi_class = 'ovr'
    return model

@st.cache_data
def load_feature_space():
    X_train = pd.read_csv(DATA_PATH)
    return X_train.astype('float64')

try:
    model = load_champion_model()
    X_background = load_feature_space()
except Exception as e:
    st.error(" Critical Error: Could not load the SmartCare clinical model or background data.")
    st.exception(e)
    st.stop()

# ============================================================
# HEADER
# ============================================================
st.title(" SmartCare AI Clinical Dashboard")
st.subheader("Disease Risk Stratification Engine (Option C)")
st.caption("Powered by Machine Learning & Shapley Additive Explanations (SHAP)")
st.divider()

# ============================================================
# SIDEBAR
# ============================================================
st.sidebar.title("System Diagnostics")
st.sidebar.write("Clinical Decision Support System (CDSS) for triaging patients into Low, Medium, or High disease risk tiers.")
st.sidebar.divider()
st.sidebar.write("**Deployed Algorithm:** Logistic Regression")
st.sidebar.write(f"**Total Features:** {X_background.shape[1]}")
st.sidebar.write("**Explainability Core:** SHAP LinearExplainer")

# ============================================================
# CLINICAL INPUT FORM
# ============================================================
st.header(" Patient Clinical Profile")

col1, col2, col3 = st.columns(3)

with col1:
    age = st.number_input("Age (Years)", min_value=0, max_value=120, value=45, step=1)
    gender = st.selectbox("Gender", ["Male", "Female"])
    blood_group = st.selectbox("Blood Group", ["A+", "A-", "B+", "B-", "AB+", "AB-", "O+", "O-"])

with col2:
    systolic_bp = st.number_input("Systolic Blood Pressure (mmHg)", min_value=50.0, max_value=250.0, value=120.0)
    diastolic_bp = st.number_input("Diastolic Blood Pressure (mmHg)", min_value=30.0, max_value=150.0, value=80.0)
    blood_sugar = st.number_input("Fasting Blood Sugar (mg/dL)", min_value=50.0, max_value=500.0, value=95.0)

with col3:
    cholesterol = st.number_input("Cholesterol (mg/dL)", min_value=50.0, max_value=500.0, value=180.0)
    bmi = st.number_input("Body Mass Index (BMI)", min_value=10.0, max_value=80.0, value=24.5)
    previous_admissions = st.number_input("Previous Hospital Admissions", min_value=0, max_value=50, value=0, step=1)

health_burden_score = st.slider("Calculated Health Burden Score", min_value=0.0, max_value=100.0, value=15.0)

st.divider()

# ============================================================
# FEATURE ENGINEERING & VECTORIZATION
# ============================================================
def construct_feature_vector():
    row = pd.DataFrame(0.0, index=[0], columns=X_background.columns)

    row.loc[0, 'age'] = age
    row.loc[0, 'systolic_bp'] = systolic_bp
    row.loc[0, 'diastolic_bp'] = diastolic_bp
    row.loc[0, 'blood_sugar_mg_dl'] = blood_sugar
    row.loc[0, 'cholesterol_mg_dl'] = cholesterol
    row.loc[0, 'bmi'] = bmi
    row.loc[0, 'previous_admissions'] = previous_admissions
    row.loc[0, 'health_burden_score'] = health_burden_score

    if f'gender_{gender}' in row.columns:
        row.loc[0, f'gender_{gender}'] = 1.0

    if f'blood_group_{blood_group}' in row.columns:
        row.loc[0, f'blood_group_{blood_group}'] = 1.0

    if bmi < 18.5: bmi_cat = 'Underweight'
    elif bmi < 25: bmi_cat = 'Normal'
    elif bmi < 30: bmi_cat = 'Overweight'
    else: bmi_cat = 'Obese'
    if f'bmi_category_{bmi_cat}' in row.columns: row.loc[0, f'bmi_category_{bmi_cat}'] = 1.0

    if blood_sugar < 100: bs_cat = 'Normal'
    elif blood_sugar < 126: bs_cat = 'Prediabetic'
    else: bs_cat = 'Diabetic'
    if f'blood_sugar_category_{bs_cat}' in row.columns: row.loc[0, f'blood_sugar_category_{bs_cat}'] = 1.0

    return row

# ============================================================
# PREDICTION PROTOCOL
# ============================================================
predict_button = st.button("Generate Diagnostic Risk Assessment", type="primary", use_container_width=True)

if predict_button:
    try:
        input_vector = construct_feature_vector()

        predicted_idx = model.predict(input_vector)[0]
        probabilities = model.predict_proba(input_vector)[0]

        risk_map = {0: 'LOW RISK', 1: 'MEDIUM RISK', 2: 'HIGH RISK'}
        final_risk = risk_map[predicted_idx]

        st.subheader("Diagnosis Result:")
        if predicted_idx == 0:
            st.success(f" Patient Status: {final_risk}")
        elif predicted_idx == 1:
            st.warning(f" Patient Status: {final_risk}")
        else:
            st.error(f" Patient Status: {final_risk}")

        st.write("### Confidence Probabilities")
        c1, c2, c3 = st.columns(3)
        c1.metric("Low Risk Probability", f"{probabilities[0]*100:.1f}%")
        c2.metric("Medium Risk Probability", f"{probabilities[1]*100:.1f}%")
        c3.metric("High Risk Probability", f"{probabilities[2]*100:.1f}%")

        st.divider()

        # ============================================================
        # SHAP LOCAL EXPLANATION (GLASS-BOX AI) - 1D Bulletproof Fix
        # ============================================================
        st.header(" Explainable AI (SHAP) - Clinical Justification")
        st.write(f"Analyzing the specific factors pushing the patient towards **{final_risk}** classification.")

        with st.spinner("Executing SHAP Force Analysis..."):
            explainer = shap.LinearExplainer(model, shap.sample(X_background, 100, random_state=42))
            shap_vals = explainer.shap_values(input_vector)

            # 1. Safely Extract the proper Array based on SHAP version
            if isinstance(shap_vals, list):
                # Older SHAP: List of Arrays
                raw_shap = shap_vals[predicted_idx][0]
            else:
                # Newer SHAP: Object or Array
                if hasattr(shap_vals, 'values'):
                    shap_vals = shap_vals.values
                shap_vals = np.array(shap_vals)

                if shap_vals.ndim == 3:
                    raw_shap = shap_vals[0, :, predicted_idx]
                elif shap_vals.ndim == 2:
                    raw_shap = shap_vals[0]
                else:
                    raw_shap = shap_vals

            # 2. Force conversion to Native 1D Python Lists (Prevents Pandas ValueError)
            class_shap_values = np.ravel(raw_shap).tolist()
            patient_vals = np.ravel(input_vector.iloc[0].values).tolist()
            feature_names = input_vector.columns.tolist()

            # 3. Build DataFrame
            impact_df = pd.DataFrame({
                "Clinical Feature": feature_names,
                "Patient's Value": patient_vals,
                "SHAP Impact": class_shap_values
            })

            impact_df['Absolute Impact'] = np.abs(impact_df['SHAP Impact'])
            top_drivers = impact_df.sort_values('Absolute Impact', ascending=False).head(10)

            top_drivers['Direction'] = top_drivers['SHAP Impact'].apply(lambda x: "Increases Risk" if x > 0 else "Decreases Risk")

            st.dataframe(top_drivers[["Clinical Feature", "Patient's Value", "SHAP Impact", "Direction"]], use_container_width=True, hide_index=True)

        st.warning(
            " **Ethical & Safety Disclaimer:**\n"
            "This CDSS prototype is developed strictly for academic demonstration. "
            "Algorithmic predictions are statistically derived and MUST NOT overrule "
            "certified medical judgment."
        )

    except Exception as e:
        st.error(" Diagnostic Pipeline Failed.")
        st.exception(e)

Overwriting /content/app.py


In [ ]:
# 5. Start the Streamlit Server in the Background
!pkill -f streamlit || true  # Kill previous instances if any

!streamlit run /content/app.py \
    --server.address=0.0.0.0 \
    --server.port=8501 \
    --server.headless=true \
    --server.enableCORS=false \
    --server.enableXsrfProtection=false \
    --browser.gatherUsageStats=false \
    > /content/streamlit.log 2>&1 &

^C


In [ ]:
# 1. Terminate any existing Streamlit or localtunnel background processes
!pkill -f streamlit
!pkill -f localtunnel

import time
from google.colab import output

print(" Starting SmartCare Clinical Dashboard. Please wait a few seconds...")

# 2. Run the Streamlit server in the background
# (CORS and XSRF protections are disabled to allow the Colab proxy to render the UI)
get_ipython().system_raw('streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false &')

# Wait for the server to fully boot up
time.sleep(5)

# 3. Generate the secure internal Colab link
url = output.eval_js("google.colab.kernel.proxyPort(8501)")

print("="*75)
print(" Dashboard is ready! Click the secure link below to open it:")
print("", url)
print("="*75)
print("Note: If you see a blank screen after clicking the link, simply refresh (F5) the browser tab.")

 Starting SmartCare Clinical Dashboard. Please wait a few seconds...
 Dashboard is ready! Click the secure link below to open it:
 https://8501-m-s-kkb-use1d2-2wx3cy2ig9wna-d.us-east1-2.prod.colab.dev
Note: If you see a blank screen after clicking the link, simply refresh (F5) the browser tab.
